##Exercise 1 — Tokenisation BERT

In [1]:
# Cellule 1 : Installation
!pip install transformers torch --quiet

In [2]:
# Cellule 2 : Tokenisation BERT
from transformers import BertTokenizer

# Charger le tokenizer BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Choisir une phrase
sentence = "Drogba scored a beautiful goal for Chelsea in London"

# Tokeniser simplement
tokens = tokenizer.tokenize(sentence)
print(f"Tokens : {tokens}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokens : ['dr', '##og', '##ba', 'scored', 'a', 'beautiful', 'goal', 'for', 'chelsea', 'in', 'london']


In [3]:
# Préparer pour le modèle (avec tokens spéciaux)
encoded = tokenizer(
    sentence,
    padding='max_length',
    truncation=True,
    max_length=20,
    return_tensors='pt'
)

In [4]:
# Voir les IDs
input_ids = encoded['input_ids'][0]
print(f"\nInput IDs : {input_ids}")


Input IDs : tensor([ 101, 2852, 8649, 3676, 3195, 1037, 3376, 3125, 2005, 9295, 1999, 2414,
         102,    0,    0,    0,    0,    0,    0,    0])


In [5]:
# Convertir les IDs en tokens pour voir les tokens spéciaux
tokens_with_special = tokenizer.convert_ids_to_tokens(input_ids)
print(f"\nTokens avec spéciaux : {tokens_with_special}")


Tokens avec spéciaux : ['[CLS]', 'dr', '##og', '##ba', 'scored', 'a', 'beautiful', 'goal', 'for', 'chelsea', 'in', 'london', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [6]:
# Identifier les tokens spéciaux
print(f"\n[CLS] token ID : {tokenizer.cls_token_id}")
print(f"[SEP] token ID : {tokenizer.sep_token_id}")
print(f"[PAD] token ID : {tokenizer.pad_token_id}")


[CLS] token ID : 101
[SEP] token ID : 102
[PAD] token ID : 0


##Exercise 2 — Analyse de sentiment avec pipeline

In [7]:
# Cellule 3 : Pipeline sentiment
from transformers import pipeline

# Créer le pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [8]:
# Tester avec des phrases
phrases = [
    "I love eating attiéké, it is delicious!",
    "This product is terrible and I hate it",
    "The service at MTN was okay, not great"
]

for phrase in phrases:
    result = sentiment_pipeline(phrase)[0]
    print(f"Texte   : {phrase}")
    print(f"Résultat: {result['label']} ({result['score']:.2%})")
    print()

Texte   : I love eating attiéké, it is delicious!
Résultat: POSITIVE (99.99%)

Texte   : This product is terrible and I hate it
Résultat: NEGATIVE (99.97%)

Texte   : The service at MTN was okay, not great
Résultat: NEGATIVE (99.76%)



##Exercise 3 — Analyseur personnalisé

In [9]:
# Cellule 4 : Classe personnalisée
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        # Initialisation tokenizer et modèle
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.labels    = ["NEGATIVE", "POSITIVE"]

    def preprocess(self, text):
        # Nettoyage basique
        text = text.strip().lower()

        # Tokenisation
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        return inputs

    def predict(self, text):
        # Prétraitement
        inputs = self.preprocess(text)

        # Prédiction sans calcul de gradient
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Softmax pour obtenir les probabilités
        probs     = torch.softmax(outputs.logits, dim=-1)
        label_idx = torch.argmax(probs).item()
        score     = probs[0][label_idx].item()

        return {
            "text":  text,
            "label": self.labels[label_idx],
            "score": score
        }


In [10]:
# Tester
analyzer = BERTSentimentAnalyzer()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [11]:
textes = [
    "I really enjoy working with AI models",
    "This is the worst experience I have ever had",
    "Abidjan is a beautiful city with amazing food"
]

for texte in textes:
    result = analyzer.predict(texte)
    print(f"Texte  : {result['text']}")
    print(f"Label  : {result['label']} ({result['score']:.2%})")
    print()

Texte  : I really enjoy working with AI models
Label  : POSITIVE (99.83%)

Texte  : This is the worst experience I have ever had
Label  : NEGATIVE (99.98%)

Texte  : Abidjan is a beautiful city with amazing food
Label  : POSITIVE (99.99%)



##Exercise 4 — NER

In [12]:
# Cellule 5 : Classe NER
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dbmdz/bert-large-cased-finetuned-conll03-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForTokenClassification.from_pretrained(model_name)

        # Labels BIO : B=Begin, I=Inside, O=Outside
        self.id2label  = self.model.config.id2label

    def recognize(self, text):
        # Tokeniser
        inputs  = self.tokenizer(text, return_tensors='pt')
        tokens  = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        # Prédiction
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Obtenir les labels prédits
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Associer chaque token à son label
        results = []
        for token, pred_id in zip(tokens, predictions):
            label = self.id2label[pred_id.item()]
            if token not in ['[CLS]', '[SEP]', '[PAD]']:
                results.append({
                    'token': token,
                    'label': label
                })

        return results


In [13]:
# Tester
ner = BERTNamedEntityRecognizer()

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
text = "Didier Drogba played for Chelsea in London and scored many goals"
entities = ner.recognize(text)

In [15]:
print("Token → Entité")
print("-" * 30)
for item in entities:
    if item['label'] != 'O':  # afficher seulement les entités trouvées
        print(f"{item['token']:15} → {item['label']}")

Token → Entité
------------------------------
Did             → I-PER
##ier           → I-PER
Dr              → I-PER
##og            → I-PER
##ba            → I-PER
Chelsea         → I-ORG
London          → I-LOC


##Exercise 5 — Tableau comparatif BERT vs GPT

#Comparaison BERT vs GPT

| Critère | BERT | GPT |
|---------|------|-----|
| Architecture | Encoder-only | Decoder-only |
| Direction de lecture | Bidirectionnelle (← →) | Gauche à droite (→) |
| Objectif principal | Compréhension | Génération |
| Entraînement | MLM (mots masqués) | CLM (mot suivant) |
| Tokens spéciaux | [CLS], [SEP], [MASK] | <\|endoftext\|> |
| Meilleur pour | Classification, NER, Q&A | Chatbot, résumé, code |
| Exemple | Détecter spam SMS | Générer réponse client |
| Faiblesse | Ne génère pas de texte | Ne comprend pas le contexte complet |

##Exercise 6 — RAG avec BERT

## BERT dans les systèmes RAG

### C'est quoi RAG ?
RAG = Retrieval Augmented Generation.
Un système qui combine recherche de documents (BERT)
et génération de réponse (GPT).

### Le rôle de BERT dans RAG

BERT transforme chaque document en un vecteur numérique
(embedding). Ce vecteur capture le sens du document.

Quand tu poses une question, BERT transforme aussi
ta question en vecteur. On compare les vecteurs pour
trouver les documents les plus proches sémantiquement.

### Les étapes du système RAG

Étape 1 — Indexation :
BERT lit tous tes documents et crée un vecteur pour chacun.
Ces vecteurs sont stockés dans une base vectorielle (ex: FAISS).

Étape 2 — Récupération :
Ta question → BERT → vecteur question
On cherche les vecteurs documents les plus proches → top 3 documents

Étape 3 — Génération :
GPT reçoit : question + top 3 documents
GPT génère une réponse basée sur ces documents

### Exemple concret Côte d'Ivoire

Ecobank a 10 000 pages de documentation interne.
Un employé pose la question : "Quelle est la procédure
pour ouvrir un compte entreprise ?"

Sans RAG : GPT invente une réponse (hallucination)
Avec RAG :
- BERT trouve les 3 pages pertinentes dans les 10 000
- GPT génère la réponse en utilisant ces 3 pages
- Réponse précise et basée sur les vrais documents

### BERT + GPT = équipe parfaite
BERT = le chercheur (trouve l'info)
GPT  = le rédacteur (explique l'info)